# NER fine-tuning on Colab (free GPU)

Mirrors `scripts/train_ner.py` / `src/mel/ner/train.py` from the project, but run here
against a free T4 GPU instead of the local CPU-only machine.

**Before running:** zip the project folder (`C:\Opensource\NLP`) excluding `.venv/`,
`data/raw`, `data/processed` cache, and `models/` — e.g. `src/`, `configs/`, `data/kb`,
`data/processed/linking_eval_set.jsonl` are enough — then upload it in the cell below.
In Colab: **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
# --only-binary=:all: forces pip to use prebuilt wheels only (no source
# builds) — avoids the "egg_info did not run successfully" failure, which
# happens when pip falls back to building some package from source and that
# build breaks on Colab's Python version. If a package genuinely has no
# wheel available, this fails with a clear "no matching distribution" error
# naming that exact package, instead of an opaque setup.py traceback.
!pip -q install --only-binary=:all: -U "datasets>=5.0" "huggingface_hub>=1.30"
!pip -q install transformers sentence-transformers accelerate seqeval SPARQLWrapper pandas pyyaml

In [ ]:
import datasets, huggingface_hub
print('datasets:', datasets.__version__)
print('huggingface_hub:', huggingface_hub.__version__)
# Expect datasets >= 5.0 and huggingface_hub >= 1.30 here.
# If this still shows 4.8.5 / 1.29.0, the pip install above ran but this
# kernel process still has the OLD modules loaded in memory — go to
# Runtime > Restart session, then re-run from the pip install cell.

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload mel_project.zip
import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/project')
os.listdir('/content/project')

In [ ]:
import sys, os
os.chdir('/content/project')
sys.path.insert(0, '/content/project/src')

import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from mel.ner.train import train

# Uses configs/ner_xlmr.yaml as-is (5 epochs, all 4 languages).
# Edit configs/ner_xlmr.yaml before this cell if you want fewer epochs for a quicker run.
train('configs/ner_xlmr.yaml')

In [ ]:
# Zip the trained checkpoint and download it, then extract into
# models/ner/xlmr-multilingual/ on your local machine.
import shutil
shutil.make_archive('/content/xlmr-multilingual', 'zip', 'models/ner/xlmr-multilingual')
files.download('/content/xlmr-multilingual.zip')